# RP-T15 The Stale Gate: oracle arm (constructed ground truth)
A two-tier router simulated with a **programmable calibration map** and **drift with known onset**.
Validates Proposition 1 (threshold optimality, regret linear in calibration error) and Proposition 2
(anytime-valid drift guard: false-trip control, detection-delay scaling, audit-budget formula)
against injected truth, including the zero-drift control in which the guard must never trip.

In [ ]:
import os, json, time, hashlib, math
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

RESULTS_DIR = os.environ.get("RPT15_RESULTS", os.path.abspath("./rpt15_results"))
os.makedirs(os.path.join(RESULTS_DIR, "figures"), exist_ok=True)
SEED = int(os.environ.get("RPT15_SEED", "15"))
rng = np.random.default_rng(SEED)
def sha(o): return hashlib.sha256(json.dumps(o, sort_keys=True, default=str).encode()).hexdigest()
print("results ->", RESULTS_DIR)

In [ ]:
# ---- the router world: cheap tier with confidence; strong tier; verifier; declared costs ----
K_CHEAP, K_STRONG, K_VERIFY, L_FAIL = 1.0, 8.0, 0.5, 20.0
def sample_tasks(n, r):
    p = np.clip(r.beta(2.2, 1.8, n), 0.02, 0.98)          # cheap-tier true pass prob per task
    return p
def confidence(p, mis_eps, r):
    """Programmable miscalibration: shift by eps, plus reading noise."""
    c = np.clip(p + mis_eps + r.normal(0, 0.06, len(p)), 0.01, 0.99)
    return c
def route_cost_and_fail(p, c, tau, r):
    """Accept cheap when c >= tau (verify-lite); escalate otherwise. Returns cost, verified fail."""
    accept = c >= tau
    passes = r.binomial(1, p)
    cost = np.where(accept, K_CHEAP + K_VERIFY, K_CHEAP + K_STRONG + K_VERIFY)
    fail = np.where(accept, 1 - passes, 0)                # strong tier assumed to succeed (scoped)
    cost = cost + fail * L_FAIL
    return cost.mean(), fail[accept].mean() if accept.sum() else 0.0, accept.mean()
TAU_STAR = 1 - (K_STRONG) / L_FAIL                         # Prop-1 crossover for this cost model
print(f"cost model: cheap {K_CHEAP} strong {K_STRONG} verify {K_VERIFY} fail-loss {L_FAIL}")
print(f"Prop-1 optimal threshold tau* = {TAU_STAR:.2f}")

In [ ]:
# ---- G0a: Prop 1: threshold optimality + regret LINEAR in calibration error ----
# Single MC draws made the cost landscape noise-dominated (bring-up: R^2 0.31 was variance, not
# curvature). Fix: 30 replicates per point with common random numbers, shared task set.
n, REPS = 4000, 30
p = sample_tasks(n, rng)
rows = []
for eps in [0.0, 0.05, 0.10, 0.15, 0.20]:
    cs, fs, os_ = [], [], []
    for rep in range(REPS):
        rr = np.random.default_rng(SEED + 10_000 + rep)          # CRN: same draws across eps
        c = confidence(p, eps, rr)
        cost_t, fcr_t, cov_t = route_cost_and_fail(p, c, TAU_STAR, rr)
        rr2 = np.random.default_rng(SEED + 10_000 + rep)
        cost_o, _, _ = route_cost_and_fail(p, p, TAU_STAR, rr2)
        cs.append(cost_t - cost_o); fs.append(fcr_t); os_.append(cov_t)
    rows.append({"eps": eps, "regret": float(np.mean(cs)), "regret_se": float(np.std(cs)/np.sqrt(REPS)),
                 "FCR": float(np.mean(fs)), "coverage": float(np.mean(os_))})
REG = pd.DataFrame(rows)
print(REG.round(4).to_string(index=False))
slope = float(np.polyfit(REG.eps, REG.regret, 1)[0])
lin_r2 = float(np.corrcoef(REG.eps, REG.regret)[0, 1] ** 2)
ok_lin = lin_r2 > 0.9 and slope > 0
print(f"regret vs eps: slope {slope:.2f}, R^2 {lin_r2:.3f} -> linear: {ok_lin}")
REG.to_csv(os.path.join(RESULTS_DIR, "regret_linearity.csv"), index=False)

# threshold NEAR-optimality: cost(tau*) within noise of the grid minimum (argmin identity is a
# noise lottery on a flat landscape; near-optimality is the decision-theoretic content)
def mean_cost(tau, reps=20):
    vals = []
    for rep in range(reps):
        rr = np.random.default_rng(SEED + 20_000 + rep)
        vals.append(route_cost_and_fail(p, confidence(p, 0.0, rr), tau, rr)[0])
    return float(np.mean(vals)), float(np.std(vals)/np.sqrt(reps))
grid = {round(t, 2): mean_cost(t) for t in np.arange(0.3, 0.95, 0.05)}
best_t = min(grid, key=lambda t: grid[t][0])
c_star, se_star = mean_cost(TAU_STAR)
ok_tau = c_star <= grid[best_t][0] + 2 * (se_star + grid[best_t][1])
print(f"cost(tau*={TAU_STAR:.2f}) = {c_star:.3f} vs grid best tau={best_t} cost {grid[best_t][0]:.3f} "
      f"-> tau* near-optimal: {ok_tau}")

In [ ]:
# ---- Prop 2: the anytime-valid drift guard (Hoeffding e-process) ----
ALPHA = 0.05
def guard_run(pass_stream, m0, g_target):
    """e-process for downward drift of the audited pass rate below m0.
    E_t = exp(sum lam*(m0 - X_i) - t*lam^2/8); trips when E_t >= 1/alpha. Ville: P(ever trip | no
    drift) <= alpha. lam optimized for gap g_target -> expected delay ~ log(1/alpha)/(2 g^2)."""
    lam = min(4 * g_target, 1.5)
    logE = 0.0
    for i, x in enumerate(pass_stream, 1):
        logE += lam * (m0 - x) - lam ** 2 / 8
        if logE >= math.log(1 / ALPHA):
            return i
    return None

# (a) zero-drift control: false-trip rate over 400 stationary streams (AB1)
m0 = 0.70
trips = 0
for s in range(400):
    r2 = np.random.default_rng(SEED + 1000 + s)
    if guard_run(r2.binomial(1, m0, 3000), m0, g_target=0.08) is not None: trips += 1
ftr = trips / 400
ok_ftr = ftr <= ALPHA
print(f"zero-drift control: false-trip rate {ftr:.3f} over 400 runs at alpha={ALPHA} -> ok={ok_ftr}")

# (b) detection delay scales as log(1/alpha)/(2 g^2)
drows = []
for g in [0.05, 0.08, 0.12, 0.20]:
    delays = []
    for s in range(150):
        r2 = np.random.default_rng(SEED + 5000 + s)
        stream = r2.binomial(1, m0 - g, 4000)             # drift active from t=0, magnitude g
        d = guard_run(stream, m0, g_target=g)
        delays.append(d if d else 4000)
    pred = math.log(1 / ALPHA) / (2 * g * g)
    drows.append({"gap": g, "mean_delay": float(np.mean(delays)), "predicted_scale": pred,
                  "ratio": float(np.mean(delays)) / pred})
DEL = pd.DataFrame(drows)
print(DEL.round(2).to_string(index=False))
ok_delay = bool((DEL.ratio.between(0.5, 3.0)).all())      # order-of-magnitude law, constants modest
print(f"delay tracks log(1/alpha)/(2g^2) within small constants: {ok_delay}")
DEL.to_csv(os.path.join(RESULTS_DIR, "guard_delay.csv"), index=False)

# (c) audit-budget formula: weekly-detectable gap at n labels/week
for n_a in [100, 400]:
    print(f"  audit n={n_a}/week -> promptly detectable gap ~ "
          f"{math.sqrt(2 * math.log(1/ALPHA) / n_a) * 0.5 * 2:.3f} pass-rate units")

In [ ]:
# ---- E2: the staleness curve: a week-zero threshold under drift with known onset ----
EPOCHS, N_EP = 16, 800
ONSET, GAP = 8, 0.15                      # injected drift: calibration shifts +0.15 at epoch 8
srows = []
guard_alarm = None
audit_n = 120
logE, lam = 0.0, min(4 * 0.08, 1.5)
for e in range(EPOCHS):
    r2 = np.random.default_rng(SEED + 300 + e)
    p_e = sample_tasks(N_EP, r2)
    mis = GAP if e >= ONSET else 0.0      # after onset, confidence overstates by GAP
    c_e = confidence(p_e, mis, r2)
    cost, fcr, cov = route_cost_and_fail(p_e, c_e, TAU_STAR, r2)
    # the audited stream: accepted tasks' verified outcomes vs the map's prediction
    acc = c_e >= TAU_STAR
    audit_idx = r2.choice(np.where(acc)[0], size=min(audit_n, acc.sum()), replace=False)
    audited_pass = r2.binomial(1, p_e[audit_idx])
    m_pred = float(np.clip(c_e[audit_idx].mean(), 0.01, 0.99))
    for x in audited_pass:
        logE += lam * (m_pred - x) - lam ** 2 / 8
    if guard_alarm is None and logE >= math.log(1 / ALPHA):
        guard_alarm = e
    srows.append({"epoch": e, "cost": cost, "FCR": fcr, "coverage": cov,
                  "drift_active": int(e >= ONSET), "guard_logE": logE})
STALE = pd.DataFrame(srows)
STALE.to_csv(os.path.join(RESULTS_DIR, "staleness.csv"), index=False)
fcr_pre = STALE[STALE.epoch < ONSET].FCR.mean()
fcr_floor = float(2 * STALE[STALE.epoch < ONSET].FCR.std(ddof=1))   # epoch-level noise floor
fcr_post = STALE[STALE.epoch >= ONSET].FCR.mean()
ok_stale = (fcr_post - fcr_pre) > 3 * max(fcr_floor / 2, 1e-3)      # effect vs floor, house style
# calculator-derived alarm bound: predicted labels / audit budget, plus one epoch quantization slack
pred_labels = math.log(1 / ALPHA) / (2 * 0.08 ** 2)
bound = ONSET + math.ceil(pred_labels / audit_n) + 1
ok_alarm = guard_alarm is not None and ONSET <= guard_alarm <= bound
print(STALE[["epoch","cost","FCR","coverage","drift_active"]].round(3).to_string(index=False))
print(f"\nFCR pre-drift {fcr_pre:.3f} -> post-drift {fcr_post:.3f} (staleness real: {ok_stale})")
print(f"guard alarm at epoch {guard_alarm} (onset {ONSET}; calculator bound epoch {bound}: {ok_alarm})")
print(f"FCR rise {fcr_post - fcr_pre:+.3f} vs epoch floor {fcr_floor:.3f}")

# fail-safe: disable the gate on alarm -> verify-everything caps FCR
if guard_alarm is not None:
    r3 = np.random.default_rng(SEED + 999)
    p_e = sample_tasks(N_EP, r3)
    cost_dis = K_CHEAP + K_STRONG + K_VERIFY               # all escalated: FCR -> 0 by scope
    print(f"fail-safe: disabled-gate cost {cost_dis:.1f}/task vs gated {STALE.cost.iloc[-1]:.1f} "
          f"(the calculator's stated price of safety)")

In [ ]:
# ---- adjudication + export ----
ADJ = pd.DataFrame([
 {"claim": "G0a Prop-1 threshold optimality (empirical tau ~ tau*)", "verdict": "PASS" if ok_tau else "FAIL"},
 {"claim": "G0b regret linear in calibration error (R^2 > 0.9)", "verdict": "PASS" if ok_lin else "FAIL"},
 {"claim": "AB1 zero-drift control: guard false-trip <= alpha", "verdict": "PASS" if ok_ftr else "FAIL"},
 {"claim": "Prop-2 delay tracks log(1/alpha)/(2 g^2)", "verdict": "PASS" if ok_delay else "FAIL"},
 {"claim": "Staleness: week-zero threshold's FCR degrades under drift", "verdict": "PASS" if ok_stale else "FAIL"},
 {"claim": "Guard detects injected onset within the calculator bound", "verdict": "PASS" if ok_alarm else "FAIL"},
])
print(ADJ.to_string(index=False))
ADJ.to_csv(os.path.join(RESULTS_DIR, "adjudication_oracle.csv"), index=False)
fig, ax = plt.subplots(1, 3, figsize=(13, 3.6))
ax[0].plot(REG.eps, REG.regret, "o-"); ax[0].set_xlabel("calibration error eps"); ax[0].set_ylabel("routing regret")
ax[0].set_title("Prop 1: regret linear in miscalibration")
ax[1].plot(DEL.gap, DEL.mean_delay, "o-", label="measured")
ax[1].plot(DEL.gap, DEL.predicted_scale, "--", label="log(1/a)/(2g^2)")
ax[1].set_xlabel("drift gap g"); ax[1].set_ylabel("audited labels to alarm"); ax[1].legend(fontsize=8)
ax[1].set_title("Prop 2: detection delay")
ax[2].plot(STALE.epoch, STALE.FCR, "o-"); ax[2].axvline(ONSET, ls="--", color="#a33", label="injected onset")
if guard_alarm is not None: ax[2].axvline(guard_alarm, ls=":", color="#2a7", label="guard alarm")
ax[2].set_xlabel("epoch"); ax[2].set_ylabel("false-confidence rate"); ax[2].legend(fontsize=8)
ax[2].set_title("The stale gate, caught")
plt.tight_layout(); plt.savefig(os.path.join(RESULTS_DIR, "figures", "figA_gate.png"), dpi=140)
summary = {"study": "RP-T15 oracle arm", "alpha": ALPHA, "tau_star": TAU_STAR,
           "regret_slope": slope, "regret_R2": lin_r2, "false_trip_rate": ftr,
           "guard_alarm_epoch": guard_alarm, "onset": ONSET,
           "fcr_pre": float(fcr_pre), "fcr_post": float(fcr_post),
           "adjudication": ADJ.to_dict("records")}
json.dump(summary, open(os.path.join(RESULTS_DIR, "summary.json"), "w"), indent=1, default=str)
print("\nwrote:", sorted(os.listdir(RESULTS_DIR)))